# KuroYs Voice Japan — Huấn luyện giọng trên Google Colab

Notebook này huấn luyện (clone) giọng tiếng Nhật **Style-Bert-VITS2 JP-Extra** cho app *KuroYs Voice Japan* trên GPU miễn phí của Colab (T4 16 GB). Phiên bản notebook: **`vj-colab-v1`**.

**Cách dùng:** trong app bấm **Tạo gói Colab** → mở notebook này bằng nút **Mở trang Colab** → chọn **Runtime → Run all** → khi ô ④ hỏi thì chọn file `.zip` app vừa tạo → chờ chạy xong → ô ⑥ tự tải file kết quả về → trong app bấm **Nhập kết quả Colab**.

Có 2 loại gói, notebook tự nhận ra:
- `vj_colab_asr_…zip` — **nhãn chữ** (chuyến 1, chỉ khi máy bạn không có môi trường nhận dạng giọng): chạy Kotoba Whisper, vài phút.
- `vj_colab_train_…zip` — **huấn luyện giọng**: khoảng 20–35 phút ở mức Tiêu chuẩn (ước lượng, chưa đo), cộng 6–10 phút cài đặt.

⚠️ **Điều khoản:** chỉ huấn luyện giọng của chính bạn hoặc giọng bạn đã được chủ sở hữu cho phép. Không dùng để mạo danh, lừa đảo hay gây hại (theo điều khoản của Style-Bert-VITS2). Âm thanh mẫu và kịch bản trong gói sẽ được tải lên máy chủ Google trong phiên Colab của bạn.

Notebook không mở web server, tunnel hay proxy. Không cần khoá API hay token.

## ① Kiểm tra GPU
Nếu báo không có GPU: **Runtime → Change runtime type → T4 GPU**, rồi chạy lại.

In [ ]:
import subprocess
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if _gpu.returncode != 0 or not _gpu.stdout.strip():
    raise SystemExit("❌ Chưa có GPU. Vào Runtime → Change runtime type → chọn T4 GPU, rồi Run all lại.")
print("✓ GPU:", _gpu.stdout.strip())

## ② Google Drive (khuyên bật khi huấn luyện)
Khi bật, checkpoint được lưu vào `MyDrive/KuroYs_VoiceJapan/` khoảng 10 phút một lần. Nếu Colab bị ngắt, mở lại notebook, **Run all** và tải lên **đúng gói cũ**: huấn luyện chạy tiếp từ checkpoint đã lưu. Mỗi checkpoint khoảng 1,3 GB (chỉ giữ bản mới nhất).

In [ ]:
LUU_VAO_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = None
if LUU_VAO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_ROOT = "/content/drive/MyDrive/KuroYs_VoiceJapan"
        print("✓ Lưu checkpoint và kết quả vào", DRIVE_ROOT)
    except Exception as error:
        DRIVE_ROOT = None
        print("⚠️ Không liên kết được Google Drive:", error)
        print("   Chạy tiếp mà không dùng Drive (Colab ngắt giữa chừng thì phải train lại từ đầu).")
        print("   Muốn dùng Drive: cho phép popup, chọn đúng tài khoản Google đang mở Colab, tích đủ quyền rồi chạy lại ô này.")
else:
    print("Không dùng Google Drive: nếu Colab ngắt giữa chừng sẽ phải huấn luyện lại từ đầu.")

## ③ Tải mã notebook (bản `vj-colab-v1`)
Bình thường mã được tải tự động từ GitHub theo tag. Nếu tag chưa được phát hành (bản chạy thử), ô này sẽ hiện nút **Choose Files**: chọn cùng lúc 6 file trong thư mục `colab_japan` của app (`vj_colab_common.py`, `vj_colab_runtime.py`, `vj_colab_asr.py`, `vj_colab_train.py`, `vj_colab_main.py`, `pins.json`).

In [ ]:
import sys, urllib.error, urllib.request
from pathlib import Path
TAG = "vj-colab-v1"
BASE = f"https://raw.githubusercontent.com/Kuroys2312/AutoMatchStudio-Releases/{TAG}/colab_japan/"
CODE = Path("/content/vj_colab")
CODE.mkdir(parents=True, exist_ok=True)
NAMES = ("vj_colab_common.py", "vj_colab_runtime.py", "vj_colab_asr.py", "vj_colab_train.py", "vj_colab_main.py", "pins.json")
try:
    for name in NAMES:
        with urllib.request.urlopen(BASE + name, timeout=60) as response:
            (CODE / name).write_bytes(response.read())
    print("✓ Đã tải mã notebook", TAG, "từ GitHub")
except urllib.error.HTTPError:
    print(f"Tag {TAG} chưa có trên GitHub (bản chạy thử): hãy chọn 6 file mã trong thư mục colab_japan của app.")
    from google.colab import files
    _code = files.upload()
    for name in NAMES:
        if name not in _code:
            raise SystemExit(f"❌ Thiếu file {name}: chạy lại ô này và chọn đủ 6 file.")
        (CODE / name).write_bytes(_code[name])
    print("✓ Đã nhận mã notebook tải lên tay")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
for _module in [name for name in sys.modules if name.startswith("vj_colab_")]:
    del sys.modules[_module]  # chạy lại ô này là dùng mã mới, không cần khởi động lại phiên
import vj_colab_common
assert vj_colab_common.NOTEBOOK_TAG == TAG, "Mã tải về không khớp phiên bản notebook"

## ④ Tải gói từ app lên
Bấm **Choose Files** và chọn file `vj_colab_…zip` mà app đã tạo (thư mục mở bằng nút trong app).

In [ ]:
from google.colab import files
from pathlib import Path
INBOX = Path("/content/vj_in")
INBOX.mkdir(parents=True, exist_ok=True)
_uploaded = files.upload()
_zips = [name for name in _uploaded if name.lower().endswith(".zip")]
if len(_zips) != 1:
    raise SystemExit("❌ Hãy chọn đúng MỘT file .zip do app tạo.")
PACKAGE = INBOX / Path(_zips[0]).name
PACKAGE.write_bytes(_uploaded[_zips[0]])
print("✓ Đã nhận gói", PACKAGE.name, f"({PACKAGE.stat().st_size / 1e6:.1f} MB)")

## ⑤ Chạy
Lần đầu trong phiên sẽ cài Python 3.11.9 và đúng các phiên bản thư viện mà app dùng, rồi tải model (kiểm tra SHA-256 từng file).

In [ ]:
import vj_colab_main
RESULT = vj_colab_main.run(PACKAGE, DRIVE_ROOT)
print("✓ Xong. File kết quả:", RESULT)

## ⑥ Tải kết quả về máy
Trình duyệt sẽ tải file `vj_colab_…_result_….zip`. Trong app bấm **Nhập kết quả Colab** và chọn file này.
- Kết quả **nhãn chữ** → app tạo tiếp **gói huấn luyện**: quay lại ô ④ (không cần cài lại), tải gói mới lên và chạy ô ④–⑥.
- Kết quả **huấn luyện** → app tự tạo bản nghe thử để bạn Chấp nhận & lưu giọng.

Nếu trình duyệt không tự tải: mở biểu tượng thư mục bên trái, vào `/content/vj_out/`, chuột phải file → Download. Khi bật Drive, bản sao kết quả huấn luyện cũng nằm trong `MyDrive/KuroYs_VoiceJapan/train/`.

In [ ]:
from google.colab import files
files.download(str(RESULT))

## Dọn dẹp (tuỳ chọn)
Sau khi app đã nhập kết quả, bạn có thể xoá thư mục gói trong `MyDrive/KuroYs_VoiceJapan/train/` để giải phóng dung lượng Drive (nhớ dọn cả Thùng rác của Drive).